## Sanskrit Machine Learning!

 Using NLP to learn embeddings from Vedic texts and explore conceptual similarity between verses, deities, and philosophical ideas (Dharma, Rta, Atman, Brahman).


### Loading the Dataset:
#### This is for the Kaggle Dataset!


Imports needed for Kaggle

In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

c:\Users\karth\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# This is the CSV file INSIDE the Kaggle dataset
file_path = "complete_rigveda_all_mandalas.json"

# Load the dataset as a pandas DataFrame
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "varunrajuvangar/rigved-all-sukta-verses-and-meaning-dataset",
    file_path,
)

# print("First 5 records:")
# print(df.head())

print("\nColumns:")
print(df.columns)

C:\Users\karth\AppData\Local\Temp\ipykernel_11924\4281516799.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(



Columns:
Index(['Mandala 1', 'Mandala 2', 'Mandala 3', 'Mandala 4', 'Mandala 5',
       'Mandala 6', 'Mandala 7', 'Mandala 8', 'Mandala 9', 'Mandala 10'],
      dtype='object')


In [6]:
# Display the first Sukta data to verify content and structure for cleaning
import pprint
first_sukta_data = df.iloc[0,0]
print("\nFirst Sukta Data:")
pprint.pprint(first_sukta_data)



First Sukta Data:
[{'padapatha': {'devanagari': {'text': 'अग्निम् । ईळे । पुरःऽहितम् । यज्ञस्य । '
                                       'देवम् । ऋत्विजम् ।होतारम् । '
                                       'रत्नऽधातमम् ॥',
                               'type': 'Padapatha Devanagari Nonaccented',
                               'words': ['अग्निम्',
                                         'ईळे',
                                         'पुरःऽहितम्',
                                         'यज्ञस्य',
                                         'देवम्',
                                         'ऋत्विजम्',
                                         'होतारम्',
                                         'रत्नऽधातमम्']},
                'transliteration': {'text': 'agním ǀ īḷe ǀ puráḥ-hitam ǀ '
                                            'yajñásya ǀ devám ǀ ṛtvíjam '
                                            'ǀhótāram ǀ ratna-dhā́tamam ǁ',
                                    'type': 'Padap

In [7]:
import pandas as pd

every_verse = []

for mandala in df.columns:
    for sukta in df.index:
        curr_cell = df.at[sukta, mandala]

        if not isinstance(curr_cell, list):
            continue

        for verse in curr_cell:
            try:
                sanskrit_verse = verse['samhita']['devanagari']['text']
                display_sanskrit = verse['sanskrit_wisdomlib']
                eng_translation = verse['translation']
                verse_num = verse['rik_number']
                
                every_verse.append({
                    'mandala': mandala,
                    'sukta': sukta,
                    'verse_num': verse_num,
                    'sanskrit_verse': sanskrit_verse,
                    'display_sanskrit': display_sanskrit,
                    'english_translation': eng_translation
                })
            
            except KeyError as e:
                print(f"KeyError for mandala {mandala}, sukta {sukta}, verse {verse.get('rik_number', '?')}: {e}")

cleaned_df = pd.DataFrame(every_verse)
print("\nCleaned DataFrame head:")
print(cleaned_df.head())


Cleaned DataFrame head:
     mandala    sukta  verse_num  \
0  Mandala 1  Sukta 1          1   
1  Mandala 1  Sukta 1          2   
2  Mandala 1  Sukta 1          3   
3  Mandala 1  Sukta 1          4   
4  Mandala 1  Sukta 1          5   

                                      sanskrit_verse  \
0  अग्निमीळे पुरोहितं यज्ञस्य देवमृत्विजं ।होतारं...   
1  अग्निः पूर्वेभिर्ऋषिभिरीड्यो नूतनैरुत ।स देवाँ...   
2  अग्निना रयिमश्नवत्पोषमेव दिवेदिवे ।यशसं वीरवत्...   
3  अग्ने यं यज्ञमध्वरं विश्वतः परिभूरसि ।स इद्देव...   
4  अग्निर्होता कविक्रतुः सत्यश्चित्रश्रवस्तमः ।दे...   

                                    display_sanskrit  \
0  अ॒ग्निमी॑ळे पु॒रोहि॑तं य॒ज्ञस्य॑ दे॒वमृ॒त्विज॑...   
1  अ॒ग्निः पूर्वे॑भि॒ॠषि॑भि॒रीड्यो॒ नूत॑नैरु॒त । ...   
2  अ॒ग्निना॑ र॒यिम॑श्नव॒त्पोष॑मे॒व दि॒वेदि॑वे । य...   
3  अग्ने॒ यं य॒ज्ञम॑ध्व॒रं वि॒श्वत॑: परि॒भूरसि॑ ।...   
4  अ॒ग्निर्होता॑ क॒विक्र॑तुः स॒त्यश्चि॒त्रश्र॑वस्...   

                                 english_translation  
0  “I glorifyAgni, the high p

## Embeddings for English Translations

In [8]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
print("Generating embeddings...")
embeddings = model.encode(
    cleaned_df['english_translation'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"Embeddings shape: {embeddings.shape}")
cleaned_df['embedding'] = list(embeddings)

Generating embeddings...


Batches: 100%|██████████| 330/330 [03:47<00:00,  1.45it/s]


Embeddings shape: (10546, 384)


In [10]:
# 1. Look at a single verse embedding
print("Single verse embedding (first 10 dimensions):")
print(embeddings[0][:10])
print(f"\nFull embedding shape for one verse: {embeddings[0].shape}")

# 2. See embeddings for first 3 verses
print("\nFirst 3 verse embeddings:")
print(embeddings[:3])

# 3. Compare two verse embeddings side-by-side
print("\nComparing verse 0 and verse 1:")
print(f"Verse 0 embedding: {embeddings[0][:5]}...")
print(f"Verse 1 embedding: {embeddings[1][:5]}...")

Single verse embedding (first 10 dimensions):
[ 0.01823254  0.6677861  -0.13582878  0.17548715 -0.11511141 -0.23859555
  0.70905596 -0.08312219  0.18871662 -0.26058328]

Full embedding shape for one verse: (384,)

First 3 verse embeddings:
[[ 0.01823254  0.6677861  -0.13582878 ... -0.1387708   0.1416574
   0.11864913]
 [ 0.15379833  0.5347147  -0.13093421 ... -0.07436576  0.17307952
   0.1142746 ]
 [ 0.11833023  0.46443495 -0.28698155 ... -0.15199092  0.15662341
   0.23451027]]

Comparing verse 0 and verse 1:
Verse 0 embedding: [ 0.01823254  0.6677861  -0.13582878  0.17548715 -0.11511141]...
Verse 1 embedding: [ 0.15379833  0.5347147  -0.13093421  0.07162705 -0.23427284]...


In [11]:
# Preparation (Do this once):
# You have your DataFrame cleaned_df.
# You have a column embeddings which contains the vectors for every verse.
# Crucial Step: Extract all those individual vectors from the DataFrame and stack them into one big block (a matrix). This makes the math fast.
verse_matrix = np.vstack(cleaned_df['embedding'].values)
print(f"\nVerse matrix shape: {verse_matrix.shape}")
print(verse_matrix[:2, :5])  

# The Search Function (Run this every time you search):
# Input: Take a text string (the "Query") from the user.
# Vectorize: Feed that text string into your SentenceTransformer model. This spits out a single vector (list of numbers).
# Math: Compare that Single Query Vector against the Big Matrix of Verse Vectors.
# Score: The result will be a list of 10,000 scores (between -1 and 1).
# Assign: Paste these scores back into your DataFrame as a new temporary column called "similarity_score".
# Sort: Sort the DataFrame so the rows with the highest "similarity_score" are at the top.
# Slice: Cut off the top 5 or 10 rows.
# Output: Print the English translation and Sanskrit text for those top rows.


Verse matrix shape: (10546, 384)
[[ 0.01823254  0.6677861  -0.13582878  0.17548715 -0.11511141]
 [ 0.15379833  0.5347147  -0.13093421  0.07162705 -0.23427284]]


In [14]:
from sklearn.metrics.pairwise import cosine_similarity
def search_verses(query, top_k):
    # Vectorize the query
    query_vector = model.encode([query])

    # Compute Similarity Scores
    similarity_scores = cosine_similarity(query_vector, verse_matrix).flatten()
    
    # Assign Scores to DataFrame
    cleaned_df['similarity_score'] = similarity_scores

    # Sort and get top K
    top_verses = cleaned_df.sort_values(by='similarity_score', ascending=False).head(top_k)

    # Output Results
    return top_verses[['mandala', 'sukta', 'verse_num', 'sanskrit_verse', 'english_translation', 'similarity_score']]

In [60]:
pd.set_option('display.max_colwidth', None)
search_verses("गणपति", top_k=10)


,mandala,sukta,verse_num,sanskrit_verse,english_translation,similarity_score
4744,Mandala 6,Sukta 41,4,सुतः सोमो असुतादिंद्र वस्यानयं श्रेयांचिकितुषे रणाय ।एतं तितिर्व उप याहि यज्ञं तेन विश्वास्तविषीरा पृणस्व ॥,“”,0.745667
9801,Mandala 10,Sukta 90,10,तस्मादश्वा अजायंत ये के चोभयादतः ।गावो ह जज्ञिरे तस्मात्तस्माज्जाता अजावयः ॥,[?],0.691046
2197,Mandala 2,Sukta 19,8,एवा ते गृत्समदाः शूर मन्मावस्यवो न वयुनानि तक्षुः ।ब्रह्मण्यंत इंद्र ते नवीय इषमूर्जं सुक्षितिं सुम्नमश्युः ॥,"“Thus, hero, have theGṛtsamadas”",0.582523
9733,Mandala 10,Sukta 87,3,None,"“Agni, the destroyer (of therākṣasas), who have two (rows of teeth), sharpening them both, applythem to (the rākṣasas, and preserve) both the upper and the lower (world); and march, radiant (Agni, againstthe rākṣasas) in the firmament, seize theyātudhānaswith your jaws.”",0.579974
1827,Mandala 1,Sukta 170,5,त्वमीशिषे वसुपते वसूनां त्वं मित्राणां मित्रपते धेष्ठः ।इंद्र त्वं मरुद्भिः सं वदस्वाध प्राशान ऋतुथा हवींषि ॥,"“(Agastya); You, Vasupati, are the lord of riches; you, Mitrapati, are the firm stay (of us), your friends; declare,Indra, along with theMaruts, (your approval of our acts), and partake of the oblation offered in due season.”",0.566372
9883,Mandala 10,Sukta 95,17,अंतरिक्षप्रां रजसो विमानीमुप शिक्षाम्युर्वशीं वसिष्ठः ।उप त्वा रातिः सुकृतस्य तिष्ठान्नि वर्तस्व हृदयं तप्यते मे ॥,"“(Purūravā). I,Vasiṣṭha, bring under subjectionŪrvaśīwho fills the firmament (with lustre) andmeasures out the rain. May (Purūravā), the bestower of the auspicious rite, abide near you; come back-- myheart is burning.”",0.560211
9720,Mandala 10,Sukta 86,13,वृषाकपायि रेवति सुपुत्र आदु सुस्नुषे ।घसत्त इंद्र उक्षणः प्रियं काचित्करं हविर्विश्वस्मादिंद्र उत्तरः ॥,"“[Vṛṣākapispeaks]: O mother of Vṛṣākapi, wealthy, possessing excellent sons, possessingexcellent daughters-in-law, letIndraeat your bulls, (give him) the beloved and most delightful ghī, Indra is aboveall (the world).”",0.557927
1970,Mandala 1,Sukta 188,11,पुरोगा अग्निर्देवानां गायत्रेण समज्यते ।स्वाहाकृतीषु रोचते ॥,"“Agni, the preceder of the gods”",0.556448
6716,Mandala 8,Sukta 32,26,अहन्वृत्रमृचीषम और्णवाभमहीशुवं ।हिमेनाविध्यदर्बुदं ॥,"“The brilliantIndraslewVṛtra, Aurṇavābha, Ahiśava; he smoteArbudawith frost.”",0.554661
6693,Mandala 8,Sukta 32,3,न्यर्बुदस्य विष्टपं वर्ष्माणं बृहतस्तिर ।कृषे तदिंद्र पौंस्यं ॥,"“Pierce the rain-holding domain of the vastArbuda; achieve,Indra, this manly exploit.”",0.552865
